# ConvMAE Pre-Training — PNG Inputs

Trains ConvMAE on radio-map PNG heatmaps (900×900 px).

**All compatibility patches pre-applied:**
- `timm==0.3.2` + `torch._six` monkey-patch for PyTorch 2.x
- `np.float` → `np.float32` (NumPy 1.24+)
- `misc.add_weight_decay` → `optim_factory.add_weight_decay`
- `torch.cuda.amp.GradScaler` / `autocast` → `torch.amp` equivalents

**Dataset:** `labels_png_v2/{training,validation,testing}`

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. GPU Check

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')

## 3. Install Dependencies

In [ ]:
# timm 0.3.2 required by ConvMAE; monkey-patch torch._six for PyTorch 2.x
!pip uninstall timm -y -q
!pip install timm==0.3.2 einops -q

import sys, collections.abc, torch
if not hasattr(torch, '_six'):
    class _Six:
        container_abcs = collections.abc
    sys.modules['torch._six'] = _Six()

import timm
print('timm version:', timm.__version__)

## 4. Clone ConvMAE & Apply Compatibility Patches

In [ ]:
import os

REPO_DIR = '/content/ConvMAE'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Alpha-VL/ConvMAE.git {REPO_DIR}
else:
    print('Already cloned — pulling.')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)

# Patch 1: np.float removed in NumPy 1.24
!sed -i 's/np\.float\b/np.float32/g' /content/ConvMAE/util/pos_embed.py

# Patch 2: add_weight_decay moved to optim_factory in timm 0.3.2
!sed -i 's/misc.add_weight_decay/optim_factory.add_weight_decay/g' /content/ConvMAE/main_pretrain.py
!sed -i '/import util.misc as misc/a import timm.optim.optim_factory as optim_factory' /content/ConvMAE/main_pretrain.py

# Patch 3: deprecated GradScaler API
!sed -i 's/torch.cuda.amp.GradScaler()/torch.amp.GradScaler("cuda")/g' /content/ConvMAE/util/misc.py

# Patch 4: deprecated autocast API
!sed -i "s/torch.cuda.amp.autocast()/torch.amp.autocast('cuda')/g" /content/ConvMAE/engine_pretrain.py

print('All patches applied.')
!ls

## 5. ⚙️ Dataset Configuration

**Edit `TOTAL_SAMPLES` to change how many images are used.**  
Examples: `1000`, `5000`, `10000`, or `None` for the full dataset.

In [ ]:
# ════════════════════════════════════════════════════
#  CHANGE THESE VALUES TO ADJUST DATASET SIZE / SPLIT
# ════════════════════════════════════════════════════
TOTAL_SAMPLES = 1000   # e.g. 1000 | 5000 | 10000 | None (full)
SPLIT_TRAIN   = 0.80
SPLIT_VAL     = 0.10
SPLIT_TEST    = 0.10
RANDOM_SEED   = 42
# ════════════════════════════════════════════════════

PNG_BASE    = '/content/drive/MyDrive/Senior Design/dataset/labels_png_v2'
DRIVE_TRAIN = f'{PNG_BASE}/training'
DRIVE_VAL   = f'{PNG_BASE}/validation'
DRIVE_TEST  = f'{PNG_BASE}/testing'

assert abs(SPLIT_TRAIN + SPLIT_VAL + SPLIT_TEST - 1.0) < 1e-6
print(f'Target samples : {TOTAL_SAMPLES if TOTAL_SAMPLES else "ALL"}')
print(f'Split          : {SPLIT_TRAIN:.0%} train / {SPLIT_VAL:.0%} val / {SPLIT_TEST:.0%} test')

## 6. Prepare Sampled ImageFolder Dataset

In [ ]:
import os, glob, random
import numpy as np

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Pool all source PNGs then sample
all_files = []
for src in [DRIVE_TRAIN, DRIVE_VAL, DRIVE_TEST]:
    all_files.extend(sorted(glob.glob(os.path.join(src, '*.png'))))

if not all_files:
    raise FileNotFoundError(f'No PNGs found under {PNG_BASE}')

random.shuffle(all_files)
if TOTAL_SAMPLES:
    all_files = all_files[:TOTAL_SAMPLES]

N       = len(all_files)
n_train = int(N * SPLIT_TRAIN)
n_val   = int(N * SPLIT_VAL)
n_test  = N - n_train - n_val

splits = {
    'train': all_files[:n_train],
    'val'  : all_files[n_train : n_train + n_val],
    'test' : all_files[n_train + n_val :],
}

DATA_ROOT = '/content/dataset_png_sampled'
for split, files in splits.items():
    cls_dir = os.path.join(DATA_ROOT, split, 'radio_map')
    os.makedirs(cls_dir, exist_ok=True)
    for f in files:
        link = os.path.join(cls_dir, os.path.basename(f))
        if not os.path.exists(link):
            os.symlink(f, link)

print(f'Total : {N}  |  Train : {n_train}  |  Val : {n_val}  |  Test : {n_test}')
print(f'Root  : {DATA_ROOT}')

## 7. Write Single-GPU Launcher

In [ ]:
launcher = '''
"""Single-GPU wrapper for ConvMAE main_pretrain.py (no DDP)."""
import os, sys
sys.path.insert(0, '/content/ConvMAE')
os.environ.setdefault('RANK', '0')
os.environ.setdefault('LOCAL_RANK', '0')
os.environ.setdefault('WORLD_SIZE', '1')
os.environ.setdefault('MASTER_ADDR', 'localhost')
os.environ.setdefault('MASTER_PORT', '12345')
import main_pretrain
'''
with open('/content/ConvMAE/main_pretrain_singlegpu.py', 'w') as f:
    f.write(launcher)
print('Launcher written.')

## 8. ⚙️ Training Hyperparameters

In [ ]:
import os

MODEL          = 'convmae_convvit_base_patch16'
INPUT_SIZE     = 224
BATCH_SIZE     = 32       # Reduce to 16 if OOM
EPOCHS         = 50       # Rapid ablation default; increase for full run
WARMUP_EPOCHS  = 5
LR             = 1.5e-4
MASK_RATIO     = 0.75
NUM_WORKERS    = 2
SAVE_CKPT_FREQ = 10

OUTPUT_DIR = '/content/drive/MyDrive/Senior Design/checkpoints/convmae_png'
LOG_DIR    = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output dir:', OUTPUT_DIR)

## 9. Run Pre-Training

In [ ]:
import subprocess, sys, os

os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12345'

cmd = [
    sys.executable, '/content/ConvMAE/main_pretrain_singlegpu.py',
    '--data_path',      DATA_ROOT,
    '--model',          MODEL,
    '--input_size',     str(INPUT_SIZE),
    '--mask_ratio',     str(MASK_RATIO),
    '--batch_size',     str(BATCH_SIZE),
    '--epochs',         str(EPOCHS),
    '--warmup_epochs',  str(WARMUP_EPOCHS),
    '--blr',            str(LR),
    '--num_workers',    str(NUM_WORKERS),
    '--pin_mem',
    '--output_dir',     OUTPUT_DIR,
    '--log_dir',        LOG_DIR,
    '--save_ckpt_freq', str(SAVE_CKPT_FREQ),
    '--dist_url',       'env://',
]

print('Command:\n ', ' '.join(cmd))
print('\n' + '='*60)

LOG_LINES_PNG = []
process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
    LOG_LINES_PNG.append(line)

process.wait()
if process.returncode != 0:
    raise RuntimeError(f'Training failed (code {process.returncode})')
print('\nTraining complete!')

## 10. Parse Training Log

In [ ]:
import json, re
import numpy as np

epochs_png, losses_png, lrs_png = [], [], []

log_path = os.path.join(OUTPUT_DIR, 'log.txt')
if os.path.exists(log_path):
    with open(log_path) as f:
        for line in f:
            try:
                d = json.loads(line.strip())
                epochs_png.append(int(d['epoch']))
                losses_png.append(float(d.get('train_loss', float('nan'))))
                lrs_png.append(float(d.get('train_lr', float('nan'))))
            except Exception:
                pass

# Fallback: extract from stdout if log.txt missing
if not epochs_png:
    for line in LOG_LINES_PNG:
        m = re.search(r'\[?(\d+)/(\d+)\]?.*?loss[:\s]+([0-9.]+)', line, re.IGNORECASE)
        if m:
            epochs_png.append(int(m.group(1)))
            losses_png.append(float(m.group(3)))
            lrs_png.append(float('nan'))

print(f'Parsed {len(epochs_png)} epoch records.')
if losses_png:
    valid = [l for l in losses_png if not np.isnan(l)]
    print(f'Loss  — min: {min(valid):.5f}  max: {max(valid):.5f}  final: {valid[-1]:.5f}')

## 11. 📊 Performance Visualisation

Six-panel dashboard: learning curve, RMSE, LR schedule, loss histogram, epoch delta, log-scale convergence.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import seaborn as sns
from matplotlib.ticker import MaxNLocator

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

if not losses_png:
    print('No training data to plot. Run training first.')
else:
    ep  = np.array(epochs_png, dtype=float)
    mse = np.array(losses_png, dtype=float)
    lrs = np.array(lrs_png,   dtype=float)
    rmse = np.sqrt(np.clip(mse, 0, None))

    # Exponential moving average for smoothing
    def ema(v, a=0.3):
        out = np.empty_like(v); out[0] = v[0]
        for i in range(1, len(v)):
            out[i] = a*v[i] + (1-a)*out[i-1]
        return out

    fig = plt.figure(figsize=(18, 12))
    fig.suptitle(
        f'ConvMAE PNG Pre-Training  |  {len(ep)} epochs  |  {n_train} train samples',
        fontsize=15, fontweight='bold', y=1.01
    )
    gs = gridspec.GridSpec(2, 3, hspace=0.42, wspace=0.35)

    # ── Panel 1: MSE / Learning curve ──────────────────────────────
    ax = fig.add_subplot(gs[0, 0])
    ax.plot(ep, mse, color='steelblue', alpha=0.35, lw=1, label='MSE raw')
    ax.plot(ep, ema(mse), color='steelblue', lw=2.5, label='MSE smoothed')
    ax.set_title('Learning Curve (MSE)'); ax.set_xlabel('Epoch'); ax.set_ylabel('MSE')
    ax.legend(); ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # ── Panel 2: RMSE ───────────────────────────────────────────────
    ax = fig.add_subplot(gs[0, 1])
    ax.plot(ep, rmse, color='tomato', lw=2.5)
    ax.fill_between(ep, rmse, alpha=0.15, color='tomato')
    ax.set_title('RMSE'); ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # ── Panel 3: Learning-rate schedule ────────────────────────────
    ax = fig.add_subplot(gs[0, 2])
    if not np.all(np.isnan(lrs)):
        ax.plot(ep, lrs, color='darkorange', lw=2.5)
        ax.set_ylabel('LR')
    else:
        ax.text(0.5, 0.5, 'LR not\nlogged', ha='center', va='center',
                transform=ax.transAxes, color='grey', fontsize=12)
    ax.set_title('LR Schedule'); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # ── Panel 4: MSE histogram ──────────────────────────────────────
    ax = fig.add_subplot(gs[1, 0])
    valid_mse = mse[~np.isnan(mse)]
    sns.histplot(valid_mse, kde=True, ax=ax, color='mediumseagreen')
    ax.set_title('MSE Distribution'); ax.set_xlabel('MSE'); ax.set_ylabel('Count')

    # ── Panel 5: Epoch-over-epoch delta ────────────────────────────
    ax = fig.add_subplot(gs[1, 1])
    if len(mse) > 1:
        d = np.diff(mse)
        ax.bar(ep[1:], d,
               color=['tomato' if x > 0 else 'steelblue' for x in d],
               edgecolor='none', alpha=0.8)
        ax.axhline(0, color='black', lw=0.8)
    ax.set_title('Epoch-over-Epoch Δ MSE'); ax.set_xlabel('Epoch'); ax.set_ylabel('Δ MSE')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # ── Panel 6: Log-scale convergence ─────────────────────────────
    ax = fig.add_subplot(gs[1, 2])
    pos_mask = mse > 0
    if pos_mask.any():
        ax.semilogy(ep[pos_mask], mse[pos_mask], color='purple', lw=2.5)
    ax.set_title('Convergence (log scale)'); ax.set_xlabel('Epoch'); ax.set_ylabel('log(MSE)')
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    plt.tight_layout()
    SAVE_FIG = os.path.join(OUTPUT_DIR, 'training_metrics_png.png')
    plt.savefig(SAVE_FIG, dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved:', SAVE_FIG)

## 12. Summary Statistics Table

In [ ]:
import pandas as pd

if losses_png:
    df = pd.DataFrame({
        'Epoch': ep.astype(int),
        'MSE'  : mse,
        'RMSE' : rmse,
        'LR'   : lrs,
    })
    print(df.describe().round(6).to_string())
    best = df.loc[df.MSE.idxmin()]
    print(f'\nBest MSE  : {best.MSE:.6f}  @ epoch {int(best.Epoch)}')
    print(f'Final MSE : {df.MSE.iloc[-1]:.6f}')
    pct = (df.MSE.iloc[0] - df.MSE.iloc[-1]) / df.MSE.iloc[0] * 100
    print(f'Total reduction: {pct:.1f}%')

    csv_path = os.path.join(OUTPUT_DIR, 'training_log_png.csv')
    df.to_csv(csv_path, index=False)
    print('CSV saved:', csv_path)

## 13. Verify Checkpoints

In [ ]:
import glob
ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.pth')))
print(f'Checkpoints ({len(ckpts)}):')
for c in ckpts:
    print(f'  {os.path.basename(c)}  ({os.path.getsize(c)/1e6:.1f} MB)')